In [1]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv("../data/processed/sales_cleaned.csv")

# Convert date column
df["date"] = pd.to_datetime(df["date"])

# Sort chronologically for each store and product
df = df.sort_values(
    ["store", "item", "date"]
).reset_index(drop=True)

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully
Shape: (913000, 4)


,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


In [2]:
# Extract calendar features

df["year"] = df["date"].dt.year

df["month"] = df["date"].dt.month

df["day"] = df["date"].dt.day

df["day_of_week"] = df["date"].dt.dayofweek

df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

df["quarter"] = df["date"].dt.quarter

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

df.head()

/tmp/ipykernel_9998/2879133836.py:9: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["day_of_week"] = df["date"].dt.dayofweek


,date,store,item,sales,year,month,day,day_of_week,week_of_year,quarter,is_weekend
0,2013-01-01,1,1,13,2013,1,1,1,1,1,0
1,2013-01-02,1,1,11,2013,1,2,2,1,1,0
2,2013-01-03,1,1,14,2013,1,3,3,1,1,0
3,2013-01-04,1,1,13,2013,1,4,4,1,1,0
4,2013-01-05,1,1,10,2013,1,5,5,1,1,1


In [ ]:
# created lag feature
# Group by store and product

grouped = df.groupby(["store", "item"])["sales"]

# Previous sales values
df["lag_1"] = grouped.shift(1)

df["lag_7"] = grouped.shift(7)

df["lag_14"] = grouped.shift(14)

df["lag_28"] = grouped.shift(28)

df[[
    "date",
    "store",
    "item",
    "sales",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28"
]].head(35)

,date,store,item,sales,lag_1,lag_7,lag_14,lag_28
0,2013-01-01,1,1,13,NaN,NaN,NaN,NaN
1,2013-01-02,1,1,11,13.0,NaN,NaN,NaN
2,2013-01-03,1,1,14,11.0,NaN,NaN,NaN
3,2013-01-04,1,1,13,14.0,NaN,NaN,NaN
4,2013-01-05,1,1,10,13.0,NaN,NaN,NaN
5,2013-01-06,1,1,12,10.0,NaN,NaN,NaN
6,2013-01-07,1,1,10,12.0,NaN,NaN,NaN
7,2013-01-08,1,1,9,10.0,13.0,NaN,NaN
8,2013-01-09,1,1,12,9.0,11.0,NaN,NaN
9,2013-01-10,1,1,9,12.0,14.0,NaN,NaN


In [4]:
# Create rolling average features
# Calculate previous 7-day average
df["rolling_mean_7"] = (
    df.groupby(["store", "item"])["sales"]
    .transform(
        lambda x: x.shift(1).rolling(window=7).mean()
    )
)

# Calculate previous 28-day average
df["rolling_mean_28"] = (
    df.groupby(["store", "item"])["sales"]
    .transform(
        lambda x: x.shift(1).rolling(window=28).mean()
    )
)

df.head(35)

,date,store,item,sales,year,month,day,day_of_week,week_of_year,quarter,is_weekend,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28
0,2013-01-01,1,1,13,2013,1,1,1,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-01-02,1,1,11,2013,1,2,2,1,1,0,13.0,NaN,NaN,NaN,NaN,NaN
2,2013-01-03,1,1,14,2013,1,3,3,1,1,0,11.0,NaN,NaN,NaN,NaN,NaN
3,2013-01-04,1,1,13,2013,1,4,4,1,1,0,14.0,NaN,NaN,NaN,NaN,NaN
4,2013-01-05,1,1,10,2013,1,5,5,1,1,1,13.0,NaN,NaN,NaN,NaN,NaN
5,2013-01-06,1,1,12,2013,1,6,6,1,1,1,10.0,NaN,NaN,NaN,NaN,NaN
6,2013-01-07,1,1,10,2013,1,7,0,2,1,0,12.0,NaN,NaN,NaN,NaN,NaN
7,2013-01-08,1,1,9,2013,1,8,1,2,1,0,10.0,13.0,NaN,NaN,11.857143,NaN
8,2013-01-09,1,1,12,2013,1,9,2,2,1,0,9.0,11.0,NaN,NaN,11.285714,NaN
9,2013-01-10,1,1,9,2013,1,10,3,2,1,0,12.0,14.0,NaN,NaN,11.428571,NaN


In [ ]:
# Display feature columns
feature_columns = [
    "date",
    "store",
    "item",
    "sales",
    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "quarter",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28"
]

df[feature_columns].head(35)

,date,store,item,sales,year,month,day,day_of_week,week_of_year,quarter,is_weekend,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28
0,2013-01-01,1,1,13,2013,1,1,1,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-01-02,1,1,11,2013,1,2,2,1,1,0,13.0,NaN,NaN,NaN,NaN,NaN
2,2013-01-03,1,1,14,2013,1,3,3,1,1,0,11.0,NaN,NaN,NaN,NaN,NaN
3,2013-01-04,1,1,13,2013,1,4,4,1,1,0,14.0,NaN,NaN,NaN,NaN,NaN
4,2013-01-05,1,1,10,2013,1,5,5,1,1,1,13.0,NaN,NaN,NaN,NaN,NaN
5,2013-01-06,1,1,12,2013,1,6,6,1,1,1,10.0,NaN,NaN,NaN,NaN,NaN
6,2013-01-07,1,1,10,2013,1,7,0,2,1,0,12.0,NaN,NaN,NaN,NaN,NaN
7,2013-01-08,1,1,9,2013,1,8,1,2,1,0,10.0,13.0,NaN,NaN,11.857143,NaN
8,2013-01-09,1,1,12,2013,1,9,2,2,1,0,9.0,11.0,NaN,NaN,11.285714,NaN
9,2013-01-10,1,1,9,2013,1,10,3,2,1,0,12.0,14.0,NaN,NaN,11.428571,NaN


In [6]:
print(df.isnull().sum())

date                   0
store                  0
item                   0
sales                  0
year                   0
month                  0
day                    0
day_of_week            0
week_of_year           0
quarter                0
is_weekend             0
lag_1                500
lag_7               3500
lag_14              7000
lag_28             14000
rolling_mean_7      3500
rolling_mean_28    14000
dtype: int64


In [ ]:
# Remove rows where the required features are unavailable:
df_model = df.dropna(
    subset=[
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28"
    ]
).copy()

print("Original shape:", df.shape)

print("After removing missing features:", df_model.shape)

Original shape: (913000, 17)
After removing missing features: (899000, 17)


In [8]:
# save the feature extracted dataset
df_model.to_csv(
    "../data/processed/sales_features.csv",
    index=False
)

print("Feature engineering completed!")
print("Saved: data/processed/sales_features.csv")

Feature engineering completed!
Saved: data/processed/sales_features.csv
